# Momentum Label Generation

This notebook generates momentum labels from raw football event data. A weighted scoring function is applied over a rolling time window to estimate which team currently has momentum.

The generated labels will serve as the target variable for training the LSTM and DistilBERT models.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

BASE_DIR = Path.cwd().parent
sys.path.append(str(BASE_DIR))

from config import *

DATA_PATH = BASE_DIR / "data" / "processed" / EXPERIMENT
DATA_PATH.mkdir(parents=True, exist_ok=True)

## Momentum Definition

Momentum is not directly observable and therefore must be approximated.

In this project, momentum is defined as the weighted influence of recent football events within a rolling five-minute window. Positive events for the home team increase the momentum score, while positive events for the away team decrease it. The final score is converted into one of three classes: Home Dominant, Balanced, or Away Dominant.

Although this is a heuristic rather than an official football metric, it provides a consistent target variable for supervised learning.

### 1. Creating Momentum Labels

Momentum is estimated by assigning a weight to each match event (e.g., goals, shots, corners, fouls). The cumulative weighted score within a rolling time window is then converted into one of three momentum classes:

- Home Dominant
- Balanced
- Away Dominant

To achieve this, we first create a helper function that maps each event type to its corresponding momentum weight before generating labels for every event sequence.

In [ ]:
df = pd.read_csv("../data/raw/archive/events.csv")

In [29]:
# Momentum weights
MOMENTUM_WEIGHTS = {
    1:  3, # Attempt
    2:  1, # Corner
    4: -1, # Yellow Card
    6: -3, # Red card
    9: -0.5 # Off-side
}


# Helper function
def get_event_weight(event):
    if event['event_type'] == 1 and event['is_goal'] == 1:
        return 5
    return MOMENTUM_WEIGHTS.get(event['event_type'], 0)

In [30]:
def calculate_momentum_label(match_df: pd.DataFrame, window_size: int=5):
    """
    Takes in a dataframe of match events & window size, returns a dataframe with momentum labels added

    Args:
        match_df (pd.DataFrame): Match events dataframe.
        window_size (int, optional): Rolling window of match time in minutes. Defaults to 5.
    """

    windows = []
    max_time = int(match_df['time'].max())

    for start in range(0, max_time, window_size):
        end = start + window_size 
        events = match_df[
            (match_df['time'] >= start) &
            (match_df['time'] < end)
        ]

        home_score = 0
        away_score = 0

        for _, event in events.iterrows():
            weight = get_event_weight(event)
            if event['side'] == 1:
                home_score += weight
            else:
                away_score += weight
        
        momentum = home_score - away_score

        # Labeling momentum
        if momentum >= 2:
            label = 0 # Home Dominant
        elif momentum <= -2:
            label = 1
        else:
            label = 2 # Balanced

        # Collect commentary text for this window
        texts = events['text'].dropna().to_list()
        combined_text = ' '.join(texts)

        # Adding it to windows list
        windows.append({
            'id_odsp': match_df['id_odsp'].iloc[0],
            'window_start': start,
            'window_end': end,
            'momentum_score': momentum,
            'label': label,
            'text' : combined_text
        })
    
    # Return windows are a dataframe
    return pd.DataFrame(windows)

In [31]:
# Testing it on one match first
match_id = df['id_odsp'].iloc[0]
match_df = df[df['id_odsp'] == match_id]

result = calculate_momentum_label(
    match_df=match_df,
    window_size=5
)

# Printout
print(result[['window_start','momentum_score', 'label', 'text']].to_string())

    window_start  momentum_score  label                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          text
0              0  

The generated labels appear consistent with the match outcome. Since Dortmund won 3–1, a larger proportion of events are labeled as **Home Dominant**, while **Balanced** and **Away Dominant** occur less frequently. This provides an initial sanity check that the weighting strategy produces reasonable momentum labels.

In [32]:
# Class distribution
print(result['label'].value_counts())
print(result['label'].value_counts(normalize=True))

label
0    8
2    6
1    4
Name: count, dtype: int64
label
0    0.444444
2    0.333333
1    0.222222
Name: proportion, dtype: float64


### 2. Applying Momentum Labels

The momentum labeling pipeline is now applied to the entire dataset. After generating labels, we perform a final quality check by removing samples with missing or empty commentary, ensuring that only valid training examples are retained.



In [33]:
# Let's apply this to all matches now
all_windows = []

for match_id in df['id_odsp'].unique():
    match_df = df[df['id_odsp'] == match_id]
    windows = calculate_momentum_label(
        match_df=match_df,
        window_size=5
    )
    all_windows.append(windows)

all_windows_df = pd.concat(all_windows, ignore_index=True)

print(all_windows_df['label'].value_counts())
print(all_windows_df['label'].value_counts(normalize=True))


label
0    62233
2    55357
1    47568
Name: count, dtype: int64
label
0    0.376809
2    0.335176
1    0.288015
Name: proportion, dtype: float64


In [ ]:
# Check length of text for every window on average
all_windows_df['text_length'] = all_windows_df['text'].str.len()
all_windows_df['text_length'].describe()

count    165158.000000
mean        413.634217
std         201.753947
min           0.000000
25%         269.000000
50%         399.000000
75%         542.000000
max        1692.000000
Name: text_length, dtype: float64

In [ ]:
# Check if there are any empty text 
print(all_windows_df['text'].isna().sum())
print((all_windows_df['text'] == '').sum())

0
2420


In [ ]:
# Since empty texts are 2.4K which is almost ~1.5% of the dataset, we can drop it

all_windows_df = all_windows_df[all_windows_df['text'].str.strip() != '']

In [ ]:
# Sanity check for empty texts
print(all_windows_df['text'].isna().sum())
print((all_windows_df['text'] == '').sum())

0
0


In [ ]:
# Check if empty text_length is removed
all_windows_df['text_length'] = all_windows_df['text'].str.len()
all_windows_df['text_length'].describe()

count    162738.000000
mean        419.785176
std         196.793956
min          24.000000
25%         275.000000
50%         403.000000
75%         544.000000
max        1692.000000
Name: text_length, dtype: float64

In [ ]:
# Check distribution once again
print(all_windows_df['label'].value_counts())
print(all_windows_df['label'].value_counts(normalize=True))

label
0    62233
2    52937
1    47568
Name: count, dtype: int64
label
0    0.382412
2    0.325290
1    0.292298
Name: proportion, dtype: float64


In [ ]:
# Save the dataframe to csv
all_windows_df.to_csv(DATA_PATH / 'momentum_labels.csv', index=False)

### 3. Creating the Train–Validation–Test Split

After preprocessing is complete, the labeled dataset is split into training, validation, and test sets. This ensures that model training, hyperparameter tuning, and final evaluation are performed on separate data partitions.

In [62]:
# Get unique ids from the dataframe
unique_ids = all_windows_df['id_odsp'].unique()

rng = np.random.default_rng(42)
shuffled_ids = rng.permutation(unique_ids)

In [63]:
n = len(shuffled_ids)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_ids = shuffled_ids[:train_end]
val_ids = shuffled_ids[train_end:val_end]
test_ids = shuffled_ids[val_end:]

In [ ]:
train_df = all_windows_df[all_windows_df['id_odsp'].isin(train_ids)]
val_df = all_windows_df[all_windows_df['id_odsp'].isin(val_ids)]
test_df = all_windows_df[all_windows_df['id_odsp'].isin(test_ids)]

print(f'The shape of Train split :{train_df.shape}')
print(f'The shape of Val split :{val_df.shape}')
print(f'The shape of Test split :{test_df.shape}')

The shape of Train split :(113914, 7)
The shape of Val split :(24367, 7)
The shape of Test split :(24457, 7)


In [ ]:
train_df.to_csv( DATA_PATH / 'train.csv', index=False)
val_df.to_csv( DATA_PATH / 'val.csv', index=False)
test_df.to_csv( DATA_PATH / 'test.csv', index=False)

## Summary

In this notebook:

- Momentum scores were generated using weighted football events.
- Scores were converted into three momentum classes.
- Invalid samples were removed.
- The cleaned dataset was split into training, validation, and test sets.

The processed data is now ready for feature extraction and model training.